# Simple MLOps Result Check

먼저 터미널에서 아래 순서가 끝나 있어야 합니다.

```bash
cd apps/simple_mlops
docker compose up -d mlflow
docker compose --profile train run --rm trainer
docker compose up -d api
docker compose --profile traffic run --rm traffic
```

In [ ]:
import json
from pathlib import Path

import pandas as pd
import requests

ROOT = Path.cwd()
DEMO = ROOT if (ROOT / "app.py").exists() else ROOT / "apps" / "simple_mlops"
API = "http://127.0.0.1:8000"

requests.get(f"{API}/health", timeout=5).json()

In [ ]:
metadata_path = DEMO / "models" / "latest_metadata.json"
metadata = json.loads(metadata_path.read_text())
pd.DataFrame([{
    "run_id": metadata["run_id"],
    "trained_at": metadata["trained_at"],
    **metadata["metrics"],
}])

In [ ]:
sample = {
    "heart_rate": 92,
    "respiratory_rate": 16,
    "body_temperature": 36.8,
    "oxygen_saturation": 95.4,
    "systolic_blood_pressure": 130,
    "diastolic_blood_pressure": 82,
}
requests.post(f"{API}/predict", json=sample, timeout=5).json()

In [ ]:
events_path = DEMO / "events" / "predictions.jsonl"
events = [json.loads(line) for line in events_path.read_text().splitlines()]
pd.DataFrame(events).tail(10)